![Title Diagram](./images/title2.png)

---

## 🧭 Agenda

This demo walks through the end-to-end process of **Reinforcement Fine-Tuning (RFT)** using the `qwen3.6-35b-a3b` reasoning model on Azure OpenAI:

1. **Dataset & Objective**  
   
2. **System Prompt & Grader Setup**  
   
3. **Preserving Evaluation Inputs (Execution Skipped)**  
   
4. **RFT Training: Setup, Launch & Monitoring**  
   
5. **Results & Completion Comparison**  
   
6. **Key Takeaways**  
   

---


## 🧮 Section 1: Dataset & Objective

In this demo, we use the **Countdown** dataset from Predibase — inspired by the UK game show where players must reach a target number using arithmetic and a limited set of numbers.

Each example contains:
- A **target number** (e.g., 97)
- A **list of four integers** (e.g., [53, 99, 45, 37])

The model’s task is to generate an arithmetic expression using those four numbers to match (or get close to) the target.

✅ Example:  
`(99 - ((53 + 37) / 45)) ≈ 97`

---

### 🔧 Additional Constraint

We make the task stricter:

> 🧩 **All four numbers must be used exactly once** in the final expression.

This encourages:
- Structured, multi-step reasoning  
- Full utilization of all inputs  
- Output formats that generalize well

---

### 🧠 Why This Is a Perfect Fit for RFT

This task is **not ideal for Supervised Fine-Tuning (SFT)**, which requires exact output labels. Here:
- There are **many valid outputs** per input
- **Evaluating correctness is easy**, but **generating the correct form is hard**
- Reward-based learning works far better than forcing the model to mimic a single path

With RFT:
- The model learns from **graded outcomes** instead of fixed answers
- We can **reinforce desired behavior** using numeric reward from a custom grader
- It's well suited for **low-data**, **high-structure** domains — like code, logic, or reasoning

---

### 🔍 Previewing the Data

Let’s load and display a few samples from the dataset to understand the format we’ll be working with:


In [ ]:
from datasets import load_dataset
import random

dataset = load_dataset("predibase/countdown", split="train")
random.seed(42)
print("Countdown dataset samples")
for index in random.sample(range(len(dataset)), 5):
    item = dataset[index]
    print(f"Target: {item['target']} Numbers: {item['nums']}")


## 🧩 Section 2: System Prompt, Response Scema & Grader Setup

To enable Reinforcement Fine-Tuning, we configure three key components: a structured prompt, a strict output schema, and a model-based grader.

![Section2 Diagram](images/md_sp_rs_cg.PNG)

---

### 💬 System Prompt

This guides the model on how to solve the Countdown-style problem and return results in a JSON structure.


In [ ]:
instruction = (
    "You are an expert in arithmetic problem solving. Given a target number and a list of numbers, "
    "combine every number exactly once using addition (+), subtraction (-), multiplication (*), "
    "or division (/) to reach the target. Use parentheses to control order of operations. "
    "Return only JSON with string fields expression and result. "
    "If an exact solution is impossible, return the closest valid result."
)


### 📦 Response Schema

To ensure the model outputs are machine-readable and consistent, we define a **response schema** that:
- Requires two fields: `"expression"` and `"result"`
- Enforces that both are strings
- Rejects completions that include extra or malformed content

This schema will be used during both **evaluation** and **fine-tuning**.


In [ ]:
response_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "math_expression",
        "strict": True,
        "schema": {
            "type": "object",
            "required": ["expression", "result"],
            "properties": {
                "expression": {"type": "string"},
                "result": {"type": "string"}
            },
            "additionalProperties": False
        }
    }
}


### 🧠 Custom Grader (Model-Based)

Instead of relying on hardcoded logic, we use a **model-based grader** (`o3-mini`) that evaluates each completion and scores it from **0 to 5** based on:
- Expression validity
- Correct usage of all numbers
- Proximity to the target result

The grader provides both a **numeric score** and a **justification**. This score becomes the reward signal during fine-tuning — enabling learning without ground truth labels.

This grader receives:
- A **target number** (as a string)
- A **list of numbers** (as a stringified list)
- A **model-generated output expression** (along with the result it claims)

The grader then:
- Parses and evaluates the arithmetic expression
- Verifies that **all numbers are used exactly once**
- Compares the evaluated result with the target
- Returns a score from **0 to 5** with reasoning

**🎯 Scoring Rubric**

| Score | Description |
|-------|-------------|
| 5     | Expression is valid, uses all numbers exactly once, and matches the target |
| 4     | Off by ±1, everything else correct |
| 3     | Off by ±2–5, with valid expression and number usage |
| 2     | Off by >5, but still valid and complete |
| 1     | Minor issues (e.g., reused or missing numbers) |
| 0     | Invalid expression or incorrect number usage |


In [ ]:
import os

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
BASE_MODEL = os.getenv("FOUNDRY_BASE_MODEL", "qwen3.6-35b-a3b")
GRADER_MODEL = "o3-mini"
print("Project endpoint:", PROJECT_ENDPOINT)
print("RFT base model:", BASE_MODEL)
print("Training type: globalStandard")
print("RFT grader model:", GRADER_MODEL)
NOTEBOOK_NAME = "demo"
JOB_SUFFIX = "qwen36-countdown-model-grader"


In [ ]:
custom_grader = {
    "name": "countdown_model_grader",
    "type": "score_model",
    "model": GRADER_MODEL,
    "input": [
        {
            "role": "developer",
            "content": (
                "Grade a Countdown answer. Verify that the JSON output has expression and result, "
                "that the expression uses each input number exactly once, uses only +, -, *, / and "
                "parentheses, that the reported result equals the expression, and that it reaches "
                "the target. Score 5 for an exact valid solution, 4 when off by 1, 3 when off by "
                "2-5, 2 when farther away but otherwise valid, 1 for a result mismatch, and 0 for "
                "invalid syntax or number usage. Return only a score and brief reasoning."
            )
        },
        {
            "role": "user",
            "content": (
                "target={{item.target}} numbers={{item.nums}} "
                "output={{sample.output_text}}"
            )
        }
    ],
    "pass_threshold": 5,
    "range": [0.0, 5.0]
}


## 📊 Section 3: Preserving Evaluation Inputs (Execution Skipped)

Before fine-tuning, we evaluate how existing models perform on the Countdown task using our custom **o3-mini-based grader**.

We compare:
- `o3` (large reasoning model)
- `qwen3.6-35b-a3b` (smaller reasoning model)
- `gpt-4.1` and `gpt-4.1-mini` (general-purpose models)

All models are evaluated on the same set of examples using the same rubric:
- Use all numbers once
- Return valid expression and result
- Get as close as possible to the target

---

### 🎯 Evaluation Setup
- We reuse the same **eval dataset** for all models
- Each model is run using the **Azure OpenAI evaluation API**
- The custom grader assigns a score from **0–5** per example

---

### 📈 What We're Measuring

| Metric              | Description                                                        |
|---------------------|--------------------------------------------------------------------|
| **Pass %**          | % of completions that passed (score ≥ threshold, e.g., 5)          |
| **Error %**         | % of completions that failed structurally (e.g., malformed output) |

---




### 📦 Step 1: Preparing Dataset for Evaluation

We use a small subset of the Countdown dataset (100 examples) for evaluation.  
Each record includes a `target`, a list of numbers (`nums`), and will later be compared to model completions using our custom grader.


In [ ]:
from pathlib import Path

eval_ready_path = "data/countdown_eval_100.jsonl"
if not Path(eval_ready_path).is_file():
    raise FileNotFoundError(eval_ready_path)
print("Preserved evaluation file (not executed):", eval_ready_path)


### ☁️ Step 2: Uploading Evaluation Dataset & Running Base Model Evaluations

We now upload the evaluation dataset to Azure OpenAI and run it against four base models:

- `o3`
- `qwen3.6-35b-a3b`
- `gpt-4.1`
- `gpt-4.1-mini`

Each run will:
- Use the same eval dataset
- Be scored by the **custom grader** (`o3-mini`)
- Output pass %, error %, and detailed scores


In [ ]:
print(
    "Evaluation upload skipped: this run is training-only and preserves the "
    "original evaluation file unchanged."
)


In [ ]:
print("Baseline inference skipped by design.")


### 🧪 Step 3: Viewing Evaluation Results

After all runs complete, we summarize and visualize the evaluation scores across models.

We compare:
- **Pass %** — completions that met the pass threshold
- **Absolute Pass %** — pass rate including error cases
- **Error %** — failed or invalid completions

This gives us a baseline to compare against our fine-tuned model.


In [ ]:
print("No deployment-based baseline metrics were collected.")


In [ ]:
print("Model grader semantics preserved; grader model:", GRADER_MODEL)


## Training-only conclusion

No deployment or inference is performed. Conclusions are limited to the validation reward curve and its initial, final, and maximum values from the terminal training result.


## 🚀 Section 4: RFT Training – Setup, Launch & Monitoring

With our system prompt, schema, and grader defined — and base model performance evaluated — we’re now ready to fine-tune `qwen3.6-35b-a3b` using **Reinforcement Fine-Tuning (RFT)**.

---

### 🧪 RFT is Sample-Efficient by Design

Unlike supervised fine-tuning, which often requires thousands of labeled examples, RFT works well with small, high-signal datasets. In this demo, we use:

- **100 training examples**
- **50 validation examples**

Each example includes:
- A full prompt (system + user message)
- Embedded `target` and `nums` fields (for use by the grader only)
- No ground-truth output — the model learns purely from **reward scores**

---

### 🔄 Workflow

1. **Prepare and convert** a small dataset into RFT-ready format
2. **Upload** the training and validation datasets to Azure OpenAI
3. **Launch** the RFT job using the Studio UI or Python SDK
4. **Monitor** training metrics like reward trends and validation scores

---

> ✅ Let’s walk through these steps now, and then evaluate the performance of our fine-tuned model.


### 🧾 Step 1: Preparing the RFT Training Dataset (Prompt + Reference Data)

Each record in the RFT dataset includes two key components:

---

#### 🗣️ `messages`: The Prompt Seen by the Model

This is the **full instruction** the model sees during training and inference. It includes:
- Task description
- Format constraints
- One or more examples
- The target and number list for that instance

This forms the basis of model behavior during completion.

---

#### 🧾 Reference Fields (`target`, `nums`)

In addition to the prompt, we include a `target` and `nums` field outside `messages`. These are:
- Used **only by the grader** during evaluation
- Sometimes redundant (they’re also embedded in the prompt), but necessary for structured scoring
- Useful for keeping model input and evaluation data logically separate

```json
{
  "messages": [
    {
      "role": "user",
      "content": "You are an expert in arithmetic problem solving... \nTarget: \"39\" Numbers: \"[65, 9, 35]\""
    }
  ],
  "target": "39",
  "nums": "[65, 9, 35]"
}


In [ ]:
import hashlib
from pathlib import Path

train_rft_path = "data/countdown_train_100.jsonl"
valid_rft_path = "data/countdown_valid_50.jsonl"

expected_files = {
    train_rft_path: {
        "records": 100,
        "bytes": 54602,
        "sha256": "2554d6a9e9aaebcd67ddbd12ad8df361fa28f769a3a2c5d271db09bf54ba23ad",
    },
    valid_rft_path: {
        "records": 50,
        "bytes": 27304,
        "sha256": "ea0326d3c53c3f5da9dc2c5836fdafd017a9168c55b7dc38b6132ed63b7cf0aa",
    },
    eval_ready_path: {
        "records": 100,
        "bytes": 5535,
        "sha256": "1f7fe8ef30a0f70fcc5db112cf385916a2a200fad702fcd0711d81818ef86413",
    },
}
for path, expected in expected_files.items():
    content = Path(path).read_bytes()
    actual = {
        "records": sum(1 for line in content.splitlines() if line.strip()),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    if actual != expected:
        raise RuntimeError(f"Preserved input changed: {path}: {actual} != {expected}")
    print(path, actual)


### ☁️ Step 2: Uploading Train and Validation Sets to Azure OpenAI

Before we launch the RFT job, we upload our training and validation datasets.

Both files are in **RFT-ready JSONL format**:
- Prompt: `messages` array with the full system + user message
- Reference: `target` and `nums` fields for evaluation
- No output — model completions will be scored dynamically using the grader

We’ll also preview a sample record to verify formatting.


In [ ]:
from scripts.io_utils import upload_file

train_file_id = await upload_file(
    "countdown_train_100-qwen36.jsonl", train_rft_path, purpose="fine-tune"
)
valid_file_id = await upload_file(
    "countdown_valid_50-qwen36.jsonl", valid_rft_path, purpose="fine-tune"
)
print("Training file ID:", train_file_id)
print("Validation file ID:", valid_file_id)


### 🚀 Step 3: Submitting the RFT Job via Azure OpenAI SDK

We now submit the RFT job using the Azure OpenAI SDK.

---

#### 🔧 Configuration Summary

- **Base Model**: `qwen3.6-35b-a3b-2025-04-16`
- **Grader**: `o3-mini` (model-based, score 0–5)
- **Response Schema**: Structured JSON format
- **Training File**: 100 examples
- **Validation File**: 50 examples

#### ⚙️ RFT Hyperparameters
- `n_epochs`: 1
- `batch_size`: 4
- `eval_interval`: 5
- `eval_samples`: 2
- `reasoning_effort`: `"medium"`
- `learning_rate_multiplier`: 2

This setup gives us a fast and focused way to validate whether RFT meaningfully improves performance — without overfitting or excessive cost.


In [ ]:
from scripts.rft_workflow import create_rft_job, wait_for_fine_tune

method = {
    "type": "reinforcement",
    "reinforcement": {
        "hyperparameters": {
            "eval_interval": 5,
            "eval_samples": 2,
            "reasoning_effort": "medium",
            "n_epochs": 1,
            "batch_size": 4,
            "learning_rate_multiplier": 2,
        },
        "grader": custom_grader,
        "response_format": response_schema,
    },
}

finetune_job = create_rft_job(
    notebook=NOTEBOOK_NAME,
    model=BASE_MODEL,
    training_file=train_file_id,
    validation_file=valid_file_id,
    method=method,
    suffix=JOB_SUFFIX,
)
finetune_job = wait_for_fine_tune(NOTEBOOK_NAME, finetune_job.id)
from scripts.rft_workflow import collect_training_evidence

job_summary = collect_training_evidence(
    NOTEBOOK_NAME,
    finetune_job,
    train_rft_path,
    valid_rft_path,
    eval_ready_path,
)
print("Terminal status:", finetune_job.status)
print("Validation reward initial:", job_summary["rewards"]["validation_reward_initial"])
print("Validation reward final:", job_summary["rewards"]["validation_reward_final"])
print("Validation reward max:", job_summary["rewards"]["validation_reward_max"])
print(job_summary["conclusion"])


### 📈 Step 4: Monitoring the RFT Job in Azure OpenAI Studio

Once the RFT job is submitted, you can monitor its progress in the Azure OpenAI Studio under the **Fine-tuning** tab.

---

**📊 Reward Curves**

- *Training Reward*: Average score (from the grader) across training completions
- *Validation Reward*: Same metric, but tracked on the held-out validation set

> Ideally, you want to see upward trends or steady plateaus — indicating the model is learning to optimize completions over time.

---

### 🚀 Step 5: Deploying the Fine-Tuned Model

After the RFT job completes successfully, the fine-tuned model must be **deployed** to make it callable via the completions or evaluation APIs.

In Azure OpenAI Studio create a new deployment using the model suffix (e.g., `ft:qwen3.6-35b-a3b-countdown-rft`)


> ✅ Once deployed, we can run evaluations using this fine-tuned model and compare results against the base models.


## 🧠 Section 5: Evaluating the Fine-Tuned Model & Comparing Results

Now that RFT is complete and our fine-tuned `ft:qwen3.6-35b-a3b` model is deployed, we evaluate it on the **same test set** and using the **same model-based grader** as before.

We'll:
1. Reuse the **existing evaluation file**
2. Run the `evaluate()` API on the fine-tuned deployment
3. Add results to our earlier comparison table

---

### ❓ Key Questions

- Did the fine-tuned model improve pass rates?
- Is it now competitive with `o3`?
- Does it maintain low error rates while using fewer resources?

> ✅ This step helps us confirm if RFT successfully aligned the model toward high-quality, constrained reasoning — without overfitting or regressions.


In [ ]:
print("Fine-tuned deployment and inference skipped by design.")


In [ ]:
print("Use the validation reward curve in job_summary for conclusions.")


In [ ]:
print("Training-only Countdown run complete.")


## Training-only conclusion

No deployment or inference is performed. Conclusions are limited to the validation reward curve and its initial, final, and maximum values from the terminal training result.


## 🎯 Key Takeaways

In this demo, we explored the power of **Reinforcement Fine-Tuning (RFT)** on a reasoning-heavy task using `qwen3.6-35b-a3b` on Azure OpenAI.

---

### ✅ What We Did

- Used the **Countdown** dataset — a structured reasoning task with multiple valid outputs
- Defined a **system prompt**, **response schema**, and a custom **model-based grader**
- Evaluated **base models** (reasoning vs non-reasoning)
- Fine-tuned `qwen3.6-35b-a3b` using **just 100 examples**
- Re-evaluated and showed **significant improvement** in both output quality and pass rate

---

### 💡 Why RFT Worked

- **No labels required** — the model learned from reward scores
- **Multiple correct answers** — RFT handled ambiguity better than SFT
- **Task constraints enforced by reward** — not hardcoded rules
- **Lightweight + fast** — trained on a small dataset with clear gains

---

### 🔁 Generalization Potential

This pattern is useful across tasks like:
- Code generation and validation
- Form-filling with strict structure
- Math, tutoring, or instruction-following
- Safety alignment (via preference or structured grading)

---

> 🎓 **Smaller models + structured tasks + clear rewards = big wins.**  
> With Azure OpenAI and RFT, it’s now easy to teach models to reason better — without needing labeled data.

![Title Diagram](./images/outro.png)
